In [1]:
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.testing import assertSchemaEqual, assertDataFrameEqual



In [2]:
if platform.system() == 'Windows':
    os.environ['PYSPARK_PYTHON'] = sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = SparkSession \
    .builder \
    .appName("Data with Nikk the Greek Spark Session") \
    .master("local[4]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [6]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze" 
}

In [7]:
class TestBronze(bronze.Bronze):
    def custom_load(self, table):
        sdf = spark.range(10).withColumn("t", F.lit(table))
        return sdf
    
bronze_instance = TestBronze(spark, **options)


In [8]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [9]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 10
+--------------------+---+------+
|         LH_BronzeTS| id|     t|
+--------------------+---+------+
|2025-01-03 04:20:...|  2|people|
|2025-01-03 04:20:...|  3|people|
|2025-01-03 04:20:...|  4|people|
|2025-01-03 04:20:...|  7|people|
|2025-01-03 04:20:...|  8|people|
|2025-01-03 04:20:...|  9|people|
|2025-01-03 04:20:...|  5|people|
|2025-01-03 04:20:...|  6|people|
|2025-01-03 04:20:...|  0|people|
|2025-01-03 04:20:...|  1|people|
+--------------------+---+------+



In [10]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Debug with Overwrite example

In [11]:
class TestSilver(silver.Silver):
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("id <= 5")
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("col", F.lit("col"))
    
silver_instance = TestSilver(spark, **options)

## 2.1 Debug the data load

In [12]:
#Debug without filter
silver_instance.load().execute("people")
actual_sdf = silver_instance.data["people"]
expected_sdf = (
    spark.range(10).withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
)
actual_sdf.show(truncate=False)
#assertSchemaEqual(actual_sdf, expected_sdf)
assertDataFrameEqual(actual_sdf.drop("LH_BronzeTS"), expected_sdf.drop("LH_BronzeTS"))


+--------------------------+---+------+
|LH_BronzeTS               |id |t     |
+--------------------------+---+------+
|2025-01-03 04:20:31.035542|2  |people|
|2025-01-03 04:20:31.035542|3  |people|
|2025-01-03 04:20:31.035542|4  |people|
|2025-01-03 04:20:31.035542|7  |people|
|2025-01-03 04:20:31.035542|8  |people|
|2025-01-03 04:20:31.035542|9  |people|
|2025-01-03 04:20:31.035542|5  |people|
|2025-01-03 04:20:31.035542|6  |people|
|2025-01-03 04:20:31.035542|0  |people|
|2025-01-03 04:20:31.035542|1  |people|
+--------------------------+---+------+



In [13]:
#Debug with filter
silver_instance.load(filter="custom").execute("people")
actual_sdf = silver_instance.data["people"]
expected_sdf = (
    spark.range(10).withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
)
actual_sdf.show(truncate=False)
#assertSchemaEqual(actual_sdf, expected_sdf)
assertDataFrameEqual(actual_sdf.drop("LH_BronzeTS"), expected_sdf.drop("LH_BronzeTS"))

+--------------------------+---+------+
|LH_BronzeTS               |id |t     |
+--------------------------+---+------+
|2025-01-03 04:20:31.035542|2  |people|
|2025-01-03 04:20:31.035542|3  |people|
|2025-01-03 04:20:31.035542|4  |people|
|2025-01-03 04:20:31.035542|5  |people|
|2025-01-03 04:20:31.035542|0  |people|
|2025-01-03 04:20:31.035542|1  |people|
+--------------------------+---+------+



# 2.2 Debug transformation

In [14]:
#Debug with default transformation
silver_instance.load(filter="custom").transform().execute("people")
actual_sdf = silver_instance.data["people"]
expected_sdf = (
    spark.range(10).withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
    .withColumn("LH_SilverTS", F.current_timestamp())
)
actual_sdf.show(truncate=False)
#assertSchemaEqual(actual_sdf, expected_sdf)
assertDataFrameEqual(actual_sdf.drop("LH_BronzeTS", "LH_SilverTS"), expected_sdf.drop("LH_BronzeTS", "LH_SilverTS"))

+--------------------------+--------------------------+---+------+---+
|LH_SilverTS               |LH_BronzeTS               |id |t     |col|
+--------------------------+--------------------------+---+------+---+
|2025-01-03 04:20:40.393066|2025-01-03 04:20:31.035542|2  |people|col|
|2025-01-03 04:20:40.393066|2025-01-03 04:20:31.035542|3  |people|col|
|2025-01-03 04:20:40.393066|2025-01-03 04:20:31.035542|4  |people|col|
|2025-01-03 04:20:40.393066|2025-01-03 04:20:31.035542|5  |people|col|
|2025-01-03 04:20:40.393066|2025-01-03 04:20:31.035542|0  |people|col|
|2025-01-03 04:20:40.393066|2025-01-03 04:20:31.035542|1  |people|col|
+--------------------------+--------------------------+---+------+---+



In [15]:
#Debug without default transformation
silver_instance.load(filter="custom").transform(ignore_defaults=True).execute("people")
actual_sdf = silver_instance.data["people"]
expected_sdf = (
    spark.range(10).withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
)
actual_sdf.show(truncate=False)
#assertSchemaEqual(actual_sdf, expected_sdf)
assertDataFrameEqual(actual_sdf.drop("LH_BronzeTS"), expected_sdf.drop("LH_BronzeTS"))

+--------------------------+---+------+---+
|LH_BronzeTS               |id |t     |col|
+--------------------------+---+------+---+
|2025-01-03 04:20:31.035542|2  |people|col|
|2025-01-03 04:20:31.035542|3  |people|col|
|2025-01-03 04:20:31.035542|4  |people|col|
|2025-01-03 04:20:31.035542|5  |people|col|
|2025-01-03 04:20:31.035542|0  |people|col|
|2025-01-03 04:20:31.035542|1  |people|col|
+--------------------------+---+------+---+



# 2.3 Debug write

In [16]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute("people")
actual_sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
expected_sdf = (
    spark.range(10).withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
    .withColumn("LH_SilverTS", F.current_timestamp())
)
actual_sdf.show(truncate=False)
#assertSchemaEqual(actual_sdf, expected_sdf)
assertDataFrameEqual(actual_sdf.drop("LH_BronzeTS", "LH_SilverTS"), expected_sdf.drop("LH_BronzeTS", "LH_SilverTS"))

+--------------------------+--------------------------+---+------+---+
|LH_SilverTS               |LH_BronzeTS               |id |t     |col|
+--------------------------+--------------------------+---+------+---+
|2025-01-03 04:20:41.472319|2025-01-03 04:20:31.035542|2  |people|col|
|2025-01-03 04:20:41.472319|2025-01-03 04:20:31.035542|3  |people|col|
|2025-01-03 04:20:41.472319|2025-01-03 04:20:31.035542|4  |people|col|
|2025-01-03 04:20:41.472319|2025-01-03 04:20:31.035542|0  |people|col|
|2025-01-03 04:20:41.472319|2025-01-03 04:20:31.035542|1  |people|col|
|2025-01-03 04:20:41.472319|2025-01-03 04:20:31.035542|5  |people|col|
+--------------------------+--------------------------+---+------+---+



# 3 Clean Up

In [17]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]